# Prophet: Getting Started Tutorial

This tutorial provides a hands-on introduction to using Prophet for predicting cellular responses to interventions.

**Prophet** is a transformer-based model that predicts cellular phenotypes based on:
- **Cell line context** (which cells you're working with)
- **Interventions** (genes/drugs applied)
- **Phenotype** (what you're measuring)

## What you'll learn:
1. How to load a pretrained Prophet model
2. What data format Prophet expects
3. How to make predictions
4. How to explore available embeddings


## 1. Setup and Imports


In [11]:
import pandas as pd
import numpy as np
from prophet import Prophet

## 2. Explore Available Models

Prophet provides several pretrained models trained on different datasets. Let's see what's available:


In [12]:
# See all available models
Prophet.list_models()


Available Prophet Model Checkpoints:

Dataset: CTRP
----------------------------------------------------------------------
  split=cell_lines      fold=0  seed=110   -> epoch=9-step=19590.ckpt
  split=cell_lines      fold=0  seed=1995  -> epoch=9.ckpt
  split=cell_lines      fold=0  seed=2024  -> epoch=9.ckpt
  split=cell_lines      fold=1  seed=110   -> epoch=9.ckpt
  split=cell_lines      fold=1  seed=1995  -> epoch=9.ckpt
  split=cell_lines      fold=1  seed=2024  -> epoch=9.ckpt
  split=cell_lines      fold=2  seed=110   -> epoch=9-step=19630.ckpt
  split=cell_lines      fold=2  seed=1995  -> epoch=9.ckpt
  split=cell_lines      fold=2  seed=2024  -> epoch=9.ckpt
  split=cell_lines      fold=3  seed=110   -> epoch=9.ckpt
  split=cell_lines      fold=3  seed=1995  -> epoch=9.ckpt
  split=cell_lines      fold=3  seed=2024  -> epoch=9.ckpt
  split=cell_lines      fold=4  seed=110   -> epoch=9.ckpt
  split=cell_lines      fold=4  seed=1995  -> epoch=9-step=19610.ckpt
  split=cell_lines

In [13]:
# Programmatic access to available configurations
configs = Prophet.available_models()
print("Available datasets:", configs['datasets'])
print("Available splits:", configs['splits'])
print("Available folds:", configs['folds'])
print("Available seeds:", configs['seeds'])


Available datasets: ['CTRP', 'GDSC', 'GDSCcomb', 'Horlbeck', 'JUMP', 'LINCS', 'SCORE', 'PRISM', 'ShifrutMarson', 'base']
Available splits: ['cell_lines', 'perturbations']
Available folds: [0, 1, 2, 3, 4]
Available seeds: [110, 1995, 2024]


## 3. Load a Pretrained Model

The easiest way to get started is with `Prophet.from_pretrained()`. This automatically downloads:
- Model checkpoint
- Cell line embeddings
- Intervention (gene/drug) embeddings


In [14]:
# Load the base pretrained model (recommended for most users)
model = Prophet.from_pretrained("base")


🔄 Downloading base model from HuggingFace Hub...
   Configuration: split=cell_lines, fold=0, seed=110
Successfully downloaded base model: epoch=29-step=45360.ckpt
  Configuration: split=unseen_cell_lines, fold=0, seed=110
  Model file: /home/icb/alejandro.tejada/.cache/prophet/base_cell_lines_fold0_seed110/datasets--theislab--Prophet/snapshots/98065de245aac12545405e16e055fe94843a247c/base_pretrained/unseen_cell_lines_fold_0/seed_110/epoch=29-step=45360.ckpt
Learning rate set to 1e-05


In [15]:
# Alternative: Load a model trained on a specific dataset
# model = Prophet.from_pretrained("GDSC", split="perturbations", fold=0, seed=110)


## 4. Understanding the Data Format

Prophet expects data in a specific tabular format with these columns:

| Column | Description | Example |
|--------|-------------|--------|
| `cell_line` | Cell line identifier | "A549", "MCF7" |
| `iv1` | Primary intervention (gene/drug) | "EGFR", "TP53" |
| `iv2` | Secondary intervention (use "negative_gene" or "negative_drug" for single interventions) | "negative_gene" |
| `phenotype` | The measurement type | "viability" |

**Important**: Cell lines and interventions must exist in Prophet's embeddings. For single interventions, use `"negative_gene"` (for gene knockouts) or `"negative_drug"` (for drug treatments) in the `iv2` column.


In [16]:
# Check what cell lines are available in the embeddings
available_cell_lines = list(model.cl_embedding.index)
print(f"Number of available cell lines: {len(available_cell_lines)}")
print(f"\nFirst 20 cell lines: {available_cell_lines[:20]}")


Number of available cell lines: 1406

First 20 cell lines: ['LC1SQSF', 'COGAR359', 'COLO794', 'KKU213', 'RT4', 'SNU283', 'NCIH1395', 'DEL', 'SNU1196', 'LC1F', '93T449', 'TGBC18TKB', 'ABC1', 'SKN', 'KE97', 'BFTC909', 'KCIMOH1', 'YKG1', 'MKN1', 'LK2']


In [17]:
# Check what interventions (genes/drugs) are available
available_interventions = list(model.iv_embedding.index)
print(f"Number of available interventions: {len(available_interventions)}")
print(f"\nFirst 20 interventions: {available_interventions[:20]}")


Number of available interventions: 153997

First 20 interventions: ['c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1', 'c/c=c1\\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o', 'c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc=c4)occn6ccocc6', 'c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=cc=n4)c(=o)n.cl', 'c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(=cc=c5)occ6=cc=cc=c6', 'c=c1c(=o)o[c@h]2[c@h]1[c@@h](oc(=o)c=c(c)c)cc1=c[c@@h](c[c@@]3(c)o[c@h]23)oc1=o', 'c=cc(=o)nc1cc(nc2nccc(-c3cn(c)c4ccccc34)n2)c(oc)cc1n(c)ccn(c)c', 'cc(=cco)c=cc=c(c)c=cc1=c(c)cccc1(c)c', 'cc(=o)c1=cc(=c(s1)sc2=c(c=c(c=c2)f)f)[n+](=o)[o-]', 'cc(=o)nc1cccc(c1)-n1c2c(c)c(=o)n(c)c(nc3ccc(i)cc3f)c2c(=o)n(c2cc2)c1=o', 'cc(=o)o[c@@h]1c2=c(c)[c@h](c[c@@](o)([c@@h](oc(=o)c3ccccc3)c3[c@@]4(co[c@@h]4c[c@h](o)[c@@]3(c)c1=o)oc(c)=o)c2(c)c)oc(=o)[c@h](o)[c@@h](nc(=o)c1ccccc1)c1ccccc1', 'cc(=o)o[c@]12co[c@@h]1c[c@h](o)[c@]1(c)[c@@h]2[c@h](oc(=o)c2ccccc2)[c@]2(o)c[c@@h](oc(=o)[c@h](o)[c@@h](nc(=o)oc(c)(c)

In [18]:
# Check what phenotypes the model was trained with
print(f"Available phenotypes: {model.phenotypes}")


Available phenotypes: ['AKAP8', 'ALDOC', 'BAG3', 'BAMBI', 'BIRC5', 'C2CD2', 'CANT1', 'CCNE2', 'CHERP', 'CNOT4', 'CSRP1', 'CTRP', 'DCTD', 'DMTF1', 'DSG2', 'ENOSF1', 'ETV1', 'FAM57A', 'FBXL12', 'FBXO11', 'FGFR4', 'GDSC', 'GDSCcomb', 'GNAI1', 'GTF2A2', 'Horlbeck', 'IGF1R', 'KDELR2', 'KIAA0753', 'MBNL2', 'MMP1', 'MOK', 'MVP', 'MYBL2', 'NENF', 'NFIL3', 'NOL3', 'NUP133', 'P4HTM', 'PARP2', 'PDIA5', 'PLEKHM1', 'PRISM', 'RALA', 'RPN1', 'RRP12', 'SACM1L', 'SCORE', 'SYNGR3', 'TATDN2', 'TCEAL4', 'TGFB3', 'TIMM17B', 'UBE2A', 'USP22', 'YKT6', 'inhouse']


## 5. Create Prediction Data

Let's create a DataFrame with experiments we want to predict. We'll pick cell lines and interventions that exist in the embeddings.


In [19]:
# Pick some cell lines and interventions from the available embeddings
sample_cell_lines = available_cell_lines[:5]
sample_interventions = available_interventions[:5]

print("Selected cell lines:", sample_cell_lines)
print("Selected interventions:", sample_interventions)


Selected cell lines: ['LC1SQSF', 'COGAR359', 'COLO794', 'KKU213', 'RT4']
Selected interventions: ['c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1', 'c/c=c1\\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o', 'c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc=c4)occn6ccocc6', 'c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=cc=n4)c(=o)n.cl', 'c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(=cc=c5)occ6=cc=cc=c6']


In [20]:
# Create prediction data: all combinations of cell lines x interventions
# Use the first available phenotype from the model
phenotype_to_use = model.phenotypes[0] if model.phenotypes else "viability"
print(f"Using phenotype: {phenotype_to_use}")

prediction_data = []
for cl in sample_cell_lines:
    for iv in sample_interventions:
        prediction_data.append({
            "cell_line": cl,
            "iv1": iv,
            "iv2": "negative_gene",  # Use "negative_gene" or "negative_drug" for single interventions
            "phenotype": phenotype_to_use
        })

df = pd.DataFrame(prediction_data)
print(f"Created {len(df)} experimental combinations to predict:")
df.head(10)


Using phenotype: AKAP8
Created 25 experimental combinations to predict:


,cell_line,iv1,iv2,phenotype
0,LC1SQSF,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,negative_gene,AKAP8
1,LC1SQSF,c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h...,negative_gene,AKAP8
2,LC1SQSF,c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc...,negative_gene,AKAP8
3,LC1SQSF,c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=...,negative_gene,AKAP8
4,LC1SQSF,c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(...,negative_gene,AKAP8
5,COGAR359,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,negative_gene,AKAP8
6,COGAR359,c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h...,negative_gene,AKAP8
7,COGAR359,c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc...,negative_gene,AKAP8
8,COGAR359,c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=...,negative_gene,AKAP8
9,COGAR359,c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(...,negative_gene,AKAP8


## 6. Make Predictions

Now let's use Prophet to predict the outcomes of these experiments:


In [21]:
# Initialize input columns (required before prediction)
model._init_input(
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype"
)

# Make predictions
predictions = model.predict(df)
predictions.head(10)



Concatenating gene embeddings: 1operation [00:00, 7194.35operation/s]

Concatenating cell line embeddings: 1operation [00:00, 9362.29operation/s]

Concatenating phenotype embeddings: 1operation [00:00, 10082.46operation/s]
/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current syste

Predicting: |          | 0/? [00:00<?, ?it/s]0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
Predicting DataLoader 0: 100%|██████████| 1/1 [00:09<00:00,  0.10it/s]


,cell_line,iv1,iv2,phenotype,pred
0,LC1SQSF,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,negative_gene,AKAP8,0.055557
1,LC1SQSF,c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h...,negative_gene,AKAP8,0.052213
2,LC1SQSF,c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc...,negative_gene,AKAP8,0.061770
3,LC1SQSF,c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=...,negative_gene,AKAP8,0.062113
4,LC1SQSF,c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(...,negative_gene,AKAP8,0.053013
5,COGAR359,c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,negative_gene,AKAP8,0.045013
6,COGAR359,c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h...,negative_gene,AKAP8,0.044293
7,COGAR359,c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc...,negative_gene,AKAP8,0.048414
8,COGAR359,c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=...,negative_gene,AKAP8,0.048515
9,COGAR359,c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(...,negative_gene,AKAP8,0.046605


In [22]:
# View prediction statistics
print("Prediction Statistics:")
print(predictions['pred'].describe())


Prediction Statistics:
count    25.000000
mean      0.048291
std       0.005947
min       0.037740
25%       0.045013
50%       0.047872
75%       0.051037
max       0.062113
Name: pred, dtype: float64


## 7. Analyze Results

Let's reshape the predictions into a more interpretable format:


In [23]:
# Pivot to create an intervention x cell line matrix
prediction_matrix = predictions.pivot(
    index='iv1',
    columns='cell_line',
    values='pred'
)

print("Predicted responses (interventions x cell lines):")
prediction_matrix


Predicted responses (interventions x cell lines):


cell_line,COGAR359,COLO794,KKU213,LC1SQSF,RT4
iv1,,,,,
c(cc1c[nh]c2ccccc12)nc1cccc(nc2ccncc2)c1,0.045013,0.046571,0.041115,0.055557,0.046966
c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o,0.044293,0.038989,0.037740,0.052213,0.049563
c1cc2=c(c(=nn2c1)c3=cc=cc=n3)c4=c5c=cc(=cc5=nc=c4)occn6ccocc6,0.048414,0.050657,0.045476,0.061770,0.051339
c1ccc(c(c1)n)nc2=nc=c(c(=n2)nc3=cc(=cc=c3)n4n=cc=n4)c(=o)n.cl,0.048515,0.050486,0.044347,0.062113,0.051037
c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(=cc=c5)occ6=cc=cc=c6,0.046605,0.047872,0.041686,0.053013,0.045921


In [24]:
# Find the intervention with lowest predicted viability for each cell line
print("Most effective intervention per cell line (lowest viability):")
for col in prediction_matrix.columns:
    best_iv = prediction_matrix[col].idxmin()
    best_val = prediction_matrix[col].min()
    print(f"  {col}: {best_iv} (predicted viability: {best_val:.3f})")


Most effective intervention per cell line (lowest viability):
  COGAR359: c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o (predicted viability: 0.044)
  COLO794: c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o (predicted viability: 0.039)
  KKU213: c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o (predicted viability: 0.038)
  LC1SQSF: c/c=c1\nc(=o)[c@h]2csscc/c=c/[c@h](cc(=o)n[c@h](c(c)c)c(=o)n2)oc(=o)[c@h](c(c)c)nc1=o (predicted viability: 0.052)
  RT4: c1ccn(c1)cc2cc(c2)n3c=c(c4=c(n=cn=c43)n)c5=cc(=cc=c5)occ6=cc=cc=c6 (predicted viability: 0.046)


## 8. Next Steps

Now that you understand the basics, here are some next steps:

### Fine-tuning on Your Data
If you have your own experimental data, you can fine-tune Prophet:
```python
model.train(
    df=your_training_data,
    val_df=your_validation_data,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    readout_col="response"
)
```

### Load Dataset-Specific Models
For specific applications, load models trained on relevant datasets:
```python
# Drug sensitivity prediction
model = Prophet.from_pretrained("GDSC")  # or "CTRP", "PRISM"

# Gene expression perturbations
model = Prophet.from_pretrained("LINCS")  # or "JUMP"

# CRISPR screening
model = Prophet.from_pretrained("Horlbeck")  # or "SCORE"
```

### Other Tutorials
- **finetuning.ipynb** - Fine-tune Prophet on your assay data
- **insilico_screening.ipynb** - Large-scale in silico screening
- **active_learning_pipeline.ipynb** - Iteratively select experiments
